In [1]:
import polars as pl
from datetime import timedelta, date, datetime
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# Анализ таргета

In [2]:
# from src.config import RAW_DATA_DIR

# events_path = RAW_DATA_DIR / "dataset/small/marketplace/events"
# #items_path = "D:/HSE_repos/dreamteam-recsys/t_ecd_data/dataset/small/marketplace/items.pq"

In [3]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/events"

In [4]:
events_df = (
    pl.scan_parquet(f"{events_dir}/*.pq", include_file_paths="path")
    .with_columns(
        pl.col("path")
          .str.extract(r"(\d+)\.pq", group_index=1)
          .cast(pl.Int32)
          .alias("day")
    )
    .sort("day")
    .drop("path")
)

In [5]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [6]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 17h 39m 51s 751152µs,67838893,"""nfmcg_15106540""","""catalog""","""view""","""android""",1082
1082d 17h 39m 51s 827007µs,89736171,"""nfmcg_12135912""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 43343µs,11916495,"""nfmcg_22992567""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 87139µs,15515933,"""nfmcg_867469""","""other""","""view""","""android""",1082
1082d 17h 39m 52s 132224µs,49878225,"""nfmcg_6159822""","""u2i""","""view""","""android""",1082


In [7]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")

Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [8]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""view""",1.034082,0.710044,1.016898
"""clickout""",268.714913,5.597366,16.392526
"""like""",3121.341812,8.046339,55.86897
"""click""",34.582149,3.571844,5.880659


In [9]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [10]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 14h 17m 602023µs,27125169,"""nfmcg_27118048""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 8s 542763µs,33507026,"""nfmcg_28041816""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 13s 595050µs,18430120,"""nfmcg_6563951""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 22s 770835µs,67452006,"""nfmcg_11449610""","""catalog""","""click""","""ios""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 25s 74028µs,54036343,"""nfmcg_8955505""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1


## Global Temporal Split

In [11]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [12]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [13]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [14]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [15]:
train, test = global_temporal_split(events_df, 0.2)

In [16]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
8918322,"[1271d 6h 49m 44s 280568µs, 1271d 7h 52m 10s 396710µs]","[""nfmcg_22479815"", ""nfmcg_22604333""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""android""]","[1271, 1271]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
85965366,"[1297d 19h 58m 27s 830955µs, 1298d 19h 1m 21s 682519µs, … 1305d 11h 59m 34s 833291µs]","[""nfmcg_631253"", ""nfmcg_21453809"", … ""nfmcg_9323944""]","[""search"", ""search"", … ""u2i""]","[""click"", ""click"", … ""clickout""]","[""ios"", ""other"", … ""android""]","[1297, 1298, … 1305]","[34.582149, 34.582149, … 268.714913]","[3.571844, 3.571844, … 5.597366]","[5.880659, 5.880659, … 16.392526]","[0, 0, … 0]","[0, 0, … 1]","[0, 0, … 0]","[1, 1, … 0]"
33913815,"[1285d 3h 55m 54s 735810µs, 1285d 3h 58m 15s 249780µs, … 1298d 13h 49m 43s 925836µs]","[""nfmcg_16905705"", ""nfmcg_25466233"", … ""nfmcg_24614961""]","[""catalog"", ""catalog"", … ""u2i""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1285, 1285, … 1298]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
42394547,"[1271d 8h 5m 59s 242856µs, 1271d 8h 14m 37s 887391µs, … 1271d 21h 6m 16s 549882µs]","[""nfmcg_16621197"", ""nfmcg_8957656"", … ""nfmcg_26091175""]","[""search"", ""search"", … ""search""]","[""clickout"", ""click"", … ""clickout""]","[""other"", ""android"", … ""android""]","[1271, 1271, … 1271]","[268.714913, 34.582149, … 268.714913]","[5.597366, 3.571844, … 5.597366]","[16.392526, 5.880659, … 16.392526]","[0, 0, … 0]","[1, 0, … 1]","[0, 0, … 0]","[0, 1, … 0]"
36901900,"[1302d 9h 18m 21s 24946µs, 1302d 11h 10m 2s 64388µs, 1302d 19h 27m 3s 122500µs]","[""nfmcg_21984785"", ""nfmcg_15328188"", ""nfmcg_28197940""]","[""u2i"", ""u2i"", ""u2i""]","[""click"", ""click"", ""click""]","[""android"", ""android"", ""android""]","[1302, 1302, 1302]","[34.582149, 34.582149, 34.582149]","[3.571844, 3.571844, 3.571844]","[5.880659, 5.880659, 5.880659]","[0, 0, 0]","[0, 0, 0]","[0, 0, 0]","[1, 1, 1]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
31360336,"[1264d 11h 3m 20s 830456µs, 1264d 11h 10m 41s 761397µs, … 1264d 20h 57m 46s 260215µs]","[""nfmcg_25024012"", ""nfmcg_10130954"", … ""nfmcg_3687095""]","[""i2i"", ""u2i"", … ""u2i""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1264, 1264, … 1264]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
45760128,"[1289d 3h 4m 3s 770411µs, 1289d 4h 41m 55s 49391µs, … 1289d 21h 43m 59s 641375µs]","[""nfmcg_20363907"", ""nfmcg_28432822"", … ""nfmcg_15753427""]","[""u2i"", ""u2i"", … ""u2i""]","[""clickout"", ""click"", … ""click""]","[""android"", ""android"", … ""other""]","[1289, 1289, … 1289]","[268.714913, 34.582149, … 34.582149]","[5.597366, 3.571844, … 3.571844]","[16.392526, 5.880659, … 5.880659]","[0, 0, … 0]","[1, 0, … 0]","[0, 0, … 0]","[0, 1, … 1]"
40349931,"[1291d 10h 50m 32s 936995µs, 1291d 11h 44m 14s 685001µs]","[""nfmcg_7640272"", ""nfmcg_23885184""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""android""]","[1291, 1291]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"


In [17]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [18]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [19]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})

In [20]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u1""",1.0
"""u2""",0.789998


# Будем реализовывать Trancated SVD

## Подготовка данных для построения матрицы взаимодействий

In [21]:
train_data = train.select(['user_id', 'item_id', 'sqrt_target']).collect(engine = 'streaming')

In [22]:
# Для того, чтобы оптимизировать использование памяти оставляем пользователей и айтемы с >= 5 взаимодействиями
MIN_INTERACTIONS = 5

user_counts = train_data.group_by('user_id').agg(pl.len().alias('user_count'))
item_counts = train_data.group_by('item_id').agg(pl.len().alias('item_count'))

In [23]:
valid_users = user_counts.filter(pl.col('user_count') >= MIN_INTERACTIONS).select('user_id')
valid_items = item_counts.filter(pl.col('item_count') >= MIN_INTERACTIONS).select('item_id')

In [24]:
train_data = (
    train_data
    .join(valid_users, on='user_id', how="inner")
    .join(valid_items, on='item_id', how="inner")
)
print(f'Взаимодействий после фильтрации: {train_data.height}')

Взаимодействий после фильтрации: 98851489


In [25]:
# Создаем таблицу уникальных id с присвоенным индексом (idx)
user_ids_unique = train_data.select(pl.col('user_id')).unique().with_row_index('user_idx') 
item_ids_unique = train_data.select(pl.col('item_id')).unique().with_row_index('item_idx')

In [26]:
# Сохраняем размеры матрицы
num_users = user_ids_unique.height
num_items = item_ids_unique.height
print(f"Размерность матрицы: {num_users} пользователей x {num_items} предметов")

Размерность матрицы: 1231601 пользователей x 577509 предметов


In [27]:
# Преобразуем в словарь
user_mapping_dict = user_ids_unique.to_dict(as_series=False)
item_mapping_dict = item_ids_unique.to_dict(as_series=False)

In [28]:
# Преобразование оригинальных id в простые числовые индексы (нужно для матрицы взаимодействий)
user_to_idx = dict(zip(user_mapping_dict['user_id'], user_mapping_dict['user_idx']))
item_to_idx = dict(zip(item_mapping_dict['item_id'], item_mapping_dict['item_idx']))

In [29]:
# Обратный переход 
idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}

In [30]:
# Проверка словаря
import itertools

for original_id, index in itertools.islice(user_to_idx.items(), 5):
    print(f'{original_id}: {index}')

31139898: 0
32803829: 1
81081476: 2
25160968: 3
68458064: 4


In [31]:
# Присоединяем созданные индексы (user_idx, item_idx)
train_data_indexed = train_data.join(user_ids_unique, on='user_id', how='left').join(item_ids_unique, on='item_id', how='left')

In [32]:
# Подготовка данных для CSR матрицы
user_indices = train_data_indexed['user_idx'].to_numpy()
item_indices = train_data_indexed['item_idx'].to_numpy()
relevance_values = train_data_indexed['sqrt_target'].to_numpy()

In [33]:
# Создание разреженной User-Item матрицы
R_train_sparse = csr_matrix(
    (relevance_values, (user_indices, item_indices)),
    shape=(num_users, num_items)
)

print(f'Размер обучающей CSR матрицы: {R_train_sparse.shape}')

Размер обучающей CSR матрицы: (1231601, 577509)


In [34]:
# Обучение TruncatedSVD
LATENT_FACTORS = 20 
svd_model = TruncatedSVD(n_components=LATENT_FACTORS, random_state=42)

# user_features = U * S
user_features = svd_model.fit_transform(R_train_sparse)
# item_features = V^T
item_features = svd_model.components_

Тут  в закомментированной части оставила то, что сначала пробовала:

In [35]:
# Так в память не поместилось

# 2.4. Прогноз: R_predicted ≈ (U*S) @ V^T
# R_predicted = user_features @ item_features
# print('Обучение и прогнозирование завершено.')

In [36]:
# Так процесс вычислений около 12-13 часов

# def get_top_k_recommendations(user_idx: int, k: int = 15):
#     """Генерирует топ-K рекомендаций для пользователя по train-индексу."""
    
#     # 1. Получаем фичи пользователя (строка user_idx из U*S)
#     user_feature_vector = user_features[user_idx]
    
#     # 2. Считаем прогноз ТОЛЬКО для этого пользователя: U_i @ V^T
#     # Это ключевой момент для экономии памяти!
#     user_predictions = user_feature_vector @ item_features 
    
#     # 3. Находим индексы предметов с наивысшими рейтингами
#     top_k_item_indices = np.argsort(user_predictions)[::-1]
    
#     # 4. Фильтруем уже взаимодействовавшие предметы
#     user_train_row = R_train_sparse.getrow(user_idx).toarray().flatten()
#     recommended_indices = [
#         idx for idx in top_k_item_indices if user_train_row[idx] == 0
#     ][:k]
    
#     # 5. Преобразование индекса обратно в ID с помощью IDX_TO_ITEM
#     top_k_item_ids = [IDX_TO_ITEM[idx] for idx in recommended_indices]
#     top_k_scores = [user_predictions[idx] for idx in recommended_indices]
    
#     return top_k_item_ids, top_k_scores

In [37]:
def get_top_k_recommendations(user_idx: int, k: int = 15):
    user_feature_vector = user_features[user_idx]
    
    # Считаем скоры 
    # Мы умножаем вектор пользователя на матрицу всех товаров
    user_predictions = user_feature_vector @ item_features 

    # Достаем из обучающей матрицы историю действий этого пользователя
    # Исключаем те товары, с которыми пользователь уже взаимодействовал (зануляем скор)
    user_train_row = R_train_sparse.getrow(user_idx).toarray().flatten()
    # Ставим -inf, чтобы они точно не попали в топ
    user_predictions[user_train_row > 0] = -np.inf
    
    # argpartition находит индексы k самых больших элементов, но не сортирует их
    # это сделано для уменьшения времени исполнения, до этого было 12-13 часов (когда argsort использовалось), сейчас уменьшилось почти до 1 часа
    ind_unsorted = np.argpartition(user_predictions, -k)[-k:]
    
    # Сортируем только эти k элементов
    ind_sorted = ind_unsorted[np.argsort(user_predictions[ind_unsorted])[::-1]]
    
    top_k_item_ids = [idx_to_item[idx] for idx in ind_sorted]
    top_k_scores = [user_predictions[idx] for idx in ind_sorted]
    
    return top_k_item_ids, top_k_scores

Замечание: в функции выше мы исключаем товары, которые пользователь взаимодействовал. Скорее всего это не совсем правильный подход. Может быть, исключать только те, которые были просмотренные, а остальные сотавлять. 

In [38]:
# Оценка на тесте
TOP_K = 15

test_data = test.select(['user_id', 'item_id', 'sqrt_target']).collect(engine='streaming')

In [39]:
# Оставляем только тех пользователей, которых SVD знает
known_users = set(user_to_idx.keys())
test_data_filtered = test_data.filter(pl.col('user_id').is_in(known_users))
print(f"Пользователей в тесте после фильтрации: {test_data_filtered.select('user_id').n_unique()}")

Пользователей в тесте после фильтрации: 635751


In [40]:
user_test_truth = (
    test_data_filtered.group_by('user_id')
    .agg(
        pl.col('item_id').alias('true_items'),
        pl.col('sqrt_target').alias('relevancy'),
    )
)

In [41]:
# Параллельная генерация рекомендаций - для того, чтобы время уменьшить
NUM_WORKERS = os.cpu_count() if os.cpu_count() is not None else 4
test_user_ids = user_test_truth['user_id'].to_numpy()
total_users = len(test_user_ids)
predicted_items_list = [None] * total_users
predicted_scores_list = [None] * total_users

# Функция-обертка для ProcessPoolExecutor
def process_user(user_id, index):
    """Генерирует рекомендации и возвращает их вместе с исходным индексом."""
    user_idx = user_to_idx[user_id] 
    top_items, top_scores = get_top_k_recommendations(user_idx, k=TOP_K)
    return index, top_items, top_scores

print(f"Начало параллельной генерации рекомендаций на {NUM_WORKERS} ядрах...")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    
    futures = [
        executor.submit(process_user, user_id, i)
        for i, user_id in enumerate(test_user_ids)
    ]
    
    # as_completed и tqdm для отслеживания прогресса
    for future in tqdm(as_completed(futures), total=total_users):
        index, top_items, top_scores = future.result()
        predicted_items_list[index] = top_items
        predicted_scores_list[index] = top_scores

print("\nПараллельная генерация рекомендаций завершена.")

Начало параллельной генерации рекомендаций на 8 ядрах...


100%|██████████████████████████████████| 635751/635751 [51:17<00:00, 206.55it/s]


Параллельная генерация рекомендаций завершена.


In [42]:
# Формирование итогового DataFrame
prediction_df = pl.DataFrame({
    'user_id': test_user_ids,
    'predicted_items': predicted_items_list,
    'predicted_scores': predicted_scores_list,
})

evaluation_df = user_test_truth.join(prediction_df, on='user_id', how='inner')

In [43]:
ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col='relevancy',
        true_items_col='true_items',
        predicted_items_col='predicted_items',
        predicted_score_col='predicted_scores',
        top_k=TOP_K,
    )
mean_ndcg = ndcg_results['ndcg'].mean()
print(f'\nСредний NDCG@{TOP_K} для SVD (k={LATENT_FACTORS}): {mean_ndcg}')


Средний NDCG@15 для SVD (k=20): 0.133126820719792
